# vLLM (inference)

A refresher on **vLLM**: a high-throughput, memory-efficient serving engine for LLM inference.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes  ·  _cross-ref domain 11 (DevOps/MLOps/Infra)_

## 1. What & Why

**vLLM** is an open-source inference and serving engine for large language models. You hand it a model
(HF Transformers weights, a local path, or a quantized checkpoint) and it gives you either an in-process
`LLM` object for batch/offline generation or an **OpenAI-compatible HTTP server** for online serving.

**The problem it solves.** Naïve autoregressive generation is brutally memory- and throughput-bound:

- The **KV-cache** (the attention keys/values for every token generated so far) dominates GPU memory at
  serving time. Pre-allocating a contiguous buffer per request for the *maximum* sequence length wastes
  60–80% of that memory to internal/external fragmentation — memory you could have spent on more
  concurrent requests.
- **Static batching** (wait for a batch, run it to completion, then start the next) leaves the GPU idle:
  short requests finish early and their slots sit empty until the longest request in the batch is done.

vLLM attacks both. **PagedAttention** stores the KV-cache in fixed-size *blocks* (like OS virtual-memory
pages) so memory is allocated on demand and shared across requests — near-zero waste. **Continuous
batching** (a.k.a. iteration-level scheduling) admits and evicts requests at every decoding *step*, so a
finished request's slot is immediately reused. Together they deliver 2–24× the throughput of vanilla
HF `generate()` at the same latency.

**Reach for it when** you need to serve an open-weights LLM at scale — a chat backend, a batch
summarization/embedding job, an offline eval over millions of prompts — and you want OpenAI-API
compatibility without writing your own scheduler.

**Don't reach for it when** you only need a few generations on CPU or a laptop (use `transformers`,
`llama.cpp`, or Ollama), when you need training/fine-tuning (vLLM is inference-only — see the TRL/QLoRA
notebooks), or when a hosted API is cheaper than running your own GPUs.

## 2. Mental Model

**Think of the GPU as an operating system, and each request as a process.**

| OS concept | vLLM analog |
|---|---|
| Physical RAM | GPU HBM reserved for the KV-cache |
| Page (4 KB) | **KV block** (e.g. 16 tokens of K/V) |
| Page table | Per-sequence **block table** mapping logical → physical blocks |
| Demand paging | Blocks allocated only as a sequence grows |
| Copy-on-write / shared pages | Shared prompt prefixes & parallel samples share blocks |
| Process scheduler | **Continuous batching** scheduler, run every decode step |

Classic serving pre-books a hotel floor for every guest in case they bring a big family, then makes the
next guest wait in the lobby until the whole floor checks out. vLLM hands out **one room at a time as you
need it**, lets unrelated guests share the hallway, and checks people in and out continuously. The result:
the building (GPU) runs near 100% occupancy instead of ~30%.

## 3. Key Concepts

- **PagedAttention** — the attention kernel that reads/writes the KV-cache through a block table instead
  of a contiguous tensor. This is what makes non-contiguous, paged KV storage possible.
- **KV block & `block_size`** — the unit of KV allocation (default 16 tokens). Smaller blocks = less
  internal fragmentation but more bookkeeping.
- **Continuous (in-flight) batching** — scheduling at the granularity of a single decode iteration, so
  the running batch changes shape every step as requests join and finish.
- **`gpu_memory_utilization`** — fraction of GPU memory vLLM may claim (default 0.9). The leftover after
  weights becomes the **KV-cache pool**; bigger pool = more concurrent sequences.
- **`max_model_len`** — max context (prompt + output) per request. Caps the block table size and bounds
  worst-case KV memory.
- **`SamplingParams`** — decoding controls: `temperature`, `top_p`, `top_k`, `max_tokens`, `n`,
  `stop`, `seed`, etc. Passed per-request.
- **Prefix caching** (`enable_prefix_caching`) — automatically reuses KV blocks for shared prompt
  prefixes (system prompts, few-shot examples) across requests.
- **Tensor / pipeline parallelism** (`tensor_parallel_size`, `pipeline_parallel_size`) — shard one model
  across multiple GPUs when it doesn't fit (or to cut latency).
- **Chunked prefill** — splits long-prompt prefill into chunks interleaved with decode so a single long
  request doesn't stall everyone else (default on in recent versions).
- **Offline vs online** — `from vllm import LLM` for in-process batch generation;
  `vllm serve <model>` for an OpenAI-compatible `/v1/chat/completions` server.

## 4. Setup

vLLM needs a **CUDA GPU** (Linux is the primary target; AMD ROCm, Intel, AWS Neuron, and CPU builds
exist but are secondary). Install matches your CUDA/torch:

```bash
# GPU (CUDA), the common case:
pip install vllm

# Pin a CUDA wheel explicitly if needed:
pip install "vllm>=0.6" --extra-index-url https://download.pytorch.org/whl/cu124
```

The runnable cells below are **CPU-friendly simulations** that teach PagedAttention and continuous
batching with plain Python — they run anywhere. The cells that actually import `vllm` and load a model
are **gated behind `os.getenv("RUN_VLLM")`** so this notebook executes top-to-bottom on a laptop with no
GPU. Set `RUN_VLLM=1` on a GPU box to exercise the real engine.

In [1]:
import os, sys

# Is the real engine importable here? (It needs a CUDA build to actually run.)
try:
    import vllm  # noqa: F401
    VLLM_AVAILABLE = True
    vllm_version = vllm.__version__
except Exception as e:  # ImportError on CPU-only / fresh kernels
    VLLM_AVAILABLE = False
    vllm_version = f"not installed ({type(e).__name__})"

RUN_VLLM = bool(os.getenv("RUN_VLLM"))  # gate heavy GPU cells

print(f"python        : {sys.version.split()[0]}")
print(f"vllm          : {vllm_version}")
print(f"RUN_VLLM gate : {RUN_VLLM}")
print("Real-engine cells will run." if (VLLM_AVAILABLE and RUN_VLLM)
      else "Real-engine cells will be skipped; CPU simulations run regardless.")

python        : 3.13.7
vllm          : not installed (ModuleNotFoundError)
RUN_VLLM gate : False
Real-engine cells will be skipped; CPU simulations run regardless.


## 5. Worked Examples

Four examples, in order of "always runs" → "needs a GPU":

1. **PagedAttention memory math** — why paging beats contiguous pre-allocation (pure Python).
2. **Continuous vs static batching** — a tiny scheduler simulation showing the throughput win (pure Python).
3. **Offline batch generation** with the real `LLM` API (gated).
4. **OpenAI-compatible server** call shape (gated / shown as code).

### Example 1 — PagedAttention memory math

Contiguous serving reserves `max_model_len` of KV for *every* concurrent request, whether or not it uses
it. Paged serving allocates in `block_size`-token blocks on demand. Let's quantify the waste for a batch
of requests with realistic, *variable* sequence lengths.

In [2]:
import numpy as np

# --- Model / serving config (Llama-3-8B-ish numbers) ---
num_layers   = 32
num_kv_heads = 8           # grouped-query attention
head_dim     = 128
dtype_bytes  = 2           # fp16/bf16
# bytes of KV per token = 2 (K and V) * layers * kv_heads * head_dim * dtype
kv_bytes_per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes

max_model_len = 8192       # what contiguous allocation must reserve per request
block_size    = 16         # vLLM KV block (tokens)

rng = np.random.default_rng(0)
# 64 concurrent requests, actual lengths far below the max (the realistic case)
actual_lens = rng.integers(100, 1500, size=64)

def gib(n): return n / 1024**3

# Contiguous: reserve the worst case for everyone.
contiguous = len(actual_lens) * max_model_len * kv_bytes_per_token

# Paged: round each request UP to a whole number of blocks, allocate only those.
blocks = np.ceil(actual_lens / block_size).astype(int)
paged = (blocks * block_size).sum() * kv_bytes_per_token

print(f"KV per token        : {kv_bytes_per_token/1024:.1f} KiB")
print(f"requests            : {len(actual_lens)}  (lengths {actual_lens.min()}-{actual_lens.max()})")
print(f"contiguous reserve  : {gib(contiguous):6.2f} GiB  (everyone booked at {max_model_len})")
print(f"paged allocation    : {gib(paged):6.2f} GiB  (block_size={block_size})")
print(f"memory saved        : {gib(contiguous - paged):6.2f} GiB  "
      f"({100*(1 - paged/contiguous):.1f}% less)")
print(f"=> ~{contiguous/paged:.1f}x more requests fit in the same KV pool")

KV per token        : 128.0 KiB
requests            : 64  (lengths 103-1496)
contiguous reserve  :  64.00 GiB  (everyone booked at 8192)
paged allocation    :   6.42 GiB  (block_size=16)
memory saved        :  57.58 GiB  (90.0% less)
=> ~10.0x more requests fit in the same KV pool


The contiguous scheme wastes the gap between each request's *actual* length and `max_model_len`. Paging
collapses that gap to at most one partially-filled block per sequence — which is exactly why vLLM can keep
far more sequences resident and thus run a much larger continuous batch.

### Example 2 — Continuous vs static batching

Static batching runs a fixed batch until *every* request in it finishes, so short requests waste GPU slots
waiting on the longest one. Continuous batching refills a freed slot at the next step. We simulate both
with a fixed pool of `slots` and count GPU-steps and slot-utilization.

In [3]:
import numpy as np

rng = np.random.default_rng(42)
n_requests = 200
slots      = 16                       # how many sequences run in parallel
# Output lengths are heavy-tailed: most short, a few very long (the realistic mix).
out_lens = rng.integers(8, 256, size=n_requests)

def static_batching(out_lens, slots):
    steps = busy = 0
    for i in range(0, len(out_lens), slots):
        batch = out_lens[i:i+slots]
        batch_steps = batch.max()       # run until the LONGEST in the batch is done
        steps += batch_steps
        busy  += batch.sum()            # token-steps actually doing useful work
    return steps, busy / (steps * slots)

def continuous_batching(out_lens, slots):
    queue = list(out_lens)
    running = []                          # remaining steps for each active slot
    steps = busy = 0
    while queue or running:
        while len(running) < slots and queue:   # backfill freed slots immediately
            running.append(queue.pop())
        steps += 1
        busy  += len(running)
        running = [r - 1 for r in running if r - 1 > 0]
    return steps, busy / (steps * slots)

s_steps, s_util = static_batching(out_lens, slots)
c_steps, c_util = continuous_batching(out_lens, slots)

print(f"requests={n_requests}, slots={slots}, total output tokens={out_lens.sum()}")
print(f"static     : {s_steps:5d} GPU-steps | slot utilization {s_util:5.1%}")
print(f"continuous : {c_steps:5d} GPU-steps | slot utilization {c_util:5.1%}")
print(f"=> continuous finishes the same work in {s_steps/c_steps:.2f}x fewer steps "
      f"({1 - c_steps/s_steps:.1%} faster)")

requests=200, slots=16, total output tokens=26733
static     :  3134 GPU-steps | slot utilization 53.3%
continuous :  1769 GPU-steps | slot utilization 94.4%
=> continuous finishes the same work in 1.77x fewer steps (43.6% faster)


Continuous batching's utilization is near 100% because no slot idles waiting on a batch-mate; static
batching's drops sharply whenever a batch mixes short and long requests. This is the throughput half of
vLLM's win (PagedAttention is the memory half).

### Example 3 — Offline batch generation (real engine, gated)

This is the canonical offline API. It only runs when `vllm` is importable **and** `RUN_VLLM=1`, so the
notebook still executes on CPU. We use a tiny model so it fits a modest GPU.

In [4]:
# Gated: needs a CUDA GPU. Set RUN_VLLM=1 to actually load a model.
if VLLM_AVAILABLE and RUN_VLLM:
    from vllm import LLM, SamplingParams

    llm = LLM(
        model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        gpu_memory_utilization=0.85,   # KV pool = this fraction minus weights
        max_model_len=2048,
        enable_prefix_caching=True,    # reuse KV for shared prompt prefixes
    )
    params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=64, seed=0)

    prompts = [
        "In one sentence, what is PagedAttention?",
        "Name three benefits of continuous batching.",
    ]
    for out in llm.generate(prompts, params):     # vLLM batches these internally
        print("PROMPT:", out.prompt)
        print("OUTPUT:", out.outputs[0].text.strip())
        print("-" * 60)
else:
    # Show the call shape without a GPU.
    print("[skipped: set RUN_VLLM=1 on a CUDA box to run the real engine]")
    print("Shape: LLM(model=...).generate(prompts, SamplingParams(...)) -> list[RequestOutput]")
    print("Each RequestOutput has .prompt and .outputs[i].text / .token_ids / .cumulative_logprob")

[skipped: set RUN_VLLM=1 on a CUDA box to run the real engine]
Shape: LLM(model=...).generate(prompts, SamplingParams(...)) -> list[RequestOutput]
Each RequestOutput has .prompt and .outputs[i].text / .token_ids / .cumulative_logprob


### Example 4 — OpenAI-compatible server

For *online* serving you don't use the `LLM` class — you launch a server and talk to it with the standard
OpenAI client. Start it from a shell:

```bash
vllm serve TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --gpu-memory-utilization 0.85 \
    --max-model-len 2048 \
    --enable-prefix-caching
# serves http://localhost:8000/v1  (chat/completions, completions, embeddings, models)
```

Then any OpenAI-compatible client works — drop-in, no code changes beyond `base_url`:

In [5]:
# Gated: needs the server running. Shown as the exact call shape regardless.
if os.getenv("VLLM_SERVER_URL"):
    from openai import OpenAI
    client = OpenAI(base_url=os.environ["VLLM_SERVER_URL"], api_key="EMPTY")
    resp = client.chat.completions.create(
        model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        messages=[{"role": "user", "content": "What does vLLM's scheduler do each step?"}],
        temperature=0.7, max_tokens=64,
    )
    print(resp.choices[0].message.content)
else:
    print("[skipped: set VLLM_SERVER_URL=http://localhost:8000/v1 after `vllm serve`]")
    print("Call shape (note api_key is a placeholder; base_url points at vLLM):")
    print('  client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")')
    print('  client.chat.completions.create(model=..., messages=[...], stream=True)')

[skipped: set VLLM_SERVER_URL=http://localhost:8000/v1 after `vllm serve`]
Call shape (note api_key is a placeholder; base_url points at vLLM):
  client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
  client.chat.completions.create(model=..., messages=[...], stream=True)


## 6. Gotchas & Pitfalls

- **OOM on startup, not on the first request.** vLLM pre-allocates the KV pool at load time from
  `gpu_memory_utilization`. If weights + pool + activations exceed VRAM you OOM immediately. Lower
  `gpu_memory_utilization` or `max_model_len`, or shard with `tensor_parallel_size`.
- **`max_model_len` too high silently shrinks the KV pool** (each sequence reserves more block-table
  budget), cutting how many requests run concurrently. Set it to what you actually need, not the model's
  theoretical max.
- **Throughput vs latency.** vLLM optimizes *aggregate throughput*. A single, latency-critical request
  with no concurrency won't beat a hand-tuned single-stream runtime like TensorRT-LLM. Measure p50/p99
  under *your* concurrency, not toy single-prompt timing.
- **Determinism isn't free.** Even with a fixed `seed`, outputs can vary across batch sizes/runs because
  continuous batching changes the reduction order of floating-point ops. Pin versions and don't assert
  exact-string equality in tests.
- **Quantization needs a matching kernel.** AWQ/GPTQ/FP8 checkpoints require the right `quantization=`
  and a GPU whose compute capability supports the kernel (e.g. FP8 needs Hopper/Ada). A mismatch is slow
  or fails to load.
- **First request after launch is slow** — CUDA-graph capture and (if enabled) prefix-cache warmup.
  Warm the server before benchmarking.
- **Chat templates matter.** `llm.generate()` takes *raw* text; for chat models apply the tokenizer's
  chat template (or use `llm.chat()` / the server's `/chat/completions`) or you'll get degenerate output.
- **It's inference-only.** No gradient updates, no fine-tuning. Train elsewhere (TRL, Unsloth, QLoRA),
  then serve the merged/adapter weights with vLLM (it supports runtime LoRA adapters via `enable_lora`).

## 7. When to Use vs Alternatives

| Tool | Best at | Trade-off vs vLLM |
|---|---|---|
| **vLLM** | High-throughput multi-tenant serving of open-weights LLMs; OpenAI-compatible API; great default | GPU-centric; not the absolute lowest single-stream latency |
| **TensorRT-LLM** | Lowest latency on NVIDIA via compiled engines | Per-model/per-GPU compile step, less flexible, NVIDIA-only |
| **Hugging Face TGI** | Production serving with the HF ecosystem & features | Historically lower throughput than vLLM; converging now |
| **SGLang** | Complex multi-turn / structured / agentic workloads (RadixAttention prefix sharing) | Newer; vLLM has broader model coverage |
| **llama.cpp / Ollama** | CPU & consumer GPUs, GGUF quants, laptops, local dev | Far lower throughput; not for large-scale serving |
| **Hosted APIs (OpenAI/Anthropic/Bedrock)** | Zero ops, frontier models, pay-per-token | No control over weights; cost at scale; data leaves your infra |
| **Plain `transformers.generate()`** | Prototyping, research, one-off generations | No paging/continuous batching → orders-of-magnitude lower serving throughput |

**Rule of thumb:** open weights + your own GPUs + many concurrent requests → vLLM is the default. Squeeze
the last millisecond of single-stream latency → TensorRT-LLM. No GPU / laptop → llama.cpp or Ollama.
Don't want to run infra → a hosted API. See the **KV-Cache**, **quantization (GPTQ/AWQ)**, and
**speculative-decoding** notebooks for techniques vLLM builds on, and domain 11 for deployment.

## 8. Resources

- **Official docs** — https://docs.vllm.ai/ (install, supported models, engine args, serving guide)
- **GitHub repo** — https://github.com/vllm-project/vllm (issues, examples, release notes)
- **PagedAttention paper** — "Efficient Memory Management for Large Language Model Serving with
  PagedAttention" (Kwon et al., SOSP 2023): https://arxiv.org/abs/2309.06180
- **Original launch blog** — https://blog.vllm.ai/2023/06/20/vllm.html (the throughput numbers & motivation)
- **OpenAI-compatible server reference** —
  https://docs.vllm.ai/en/latest/serving/openai_compatible_server.html
- **Continuous batching explainer (Anyscale)** —
  https://www.anyscale.com/blog/continuous-batching-llm-inference